# NLP Practical 5 — Semantic Analysis

Run in **Google Colab**. Cells with `pip install` only need to run once per session.

**Covers:** FOL and frame-style meaning representation, WordNet lexical relations, a typology of ambiguity with examples, and Word Sense Disambiguation via hand-built Lesk, NLTK's Lesk, and a Naive Bayes sense classifier.


In [ ]:
!pip install nltk scikit-learn -q
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")


In [ ]:
# ============================================================
# PART A: First-Order Logic + Frame-style meaning representation
# ============================================================

# FOL-style: "Every student who studies NLP will pass"
fol_example = "forall x. (student(x) & studies(x, nlp)) -> pass(x)"
print("FOL representation:", fol_example)

# Frame-style representation for: "The bank approved the loan"
frame = {
    "frame": "Approval",
    "roles": {
        "Approver": "the bank",
        "Decision": "approved",
        "Item": "the loan"
    }
}
print("\nFrame representation:", frame)


In [ ]:
# ============================================================
# PART B: WordNet lexical relations
# ============================================================
from nltk.corpus import wordnet as wn

word = "bank"
print(f"Senses of '{word}':")
for s in wn.synsets(word):
    print(" -", s.name(), ":", s.definition())

dog = wn.synset("dog.n.01")
print("\nHypernyms of dog:", [h.name() for h in dog.hypernyms()])
print("Hyponyms of dog (sample):", [h.name() for h in dog.hyponyms()[:5]])
print("Synonyms (lemmas) of 'happy':", [l.name() for s in wn.synsets("happy") for l in s.lemmas()][:8])


In [ ]:
# ============================================================
# PART C: Ambiguity typology
# ============================================================
examples = {
    "Lexical ambiguity": "I went to the BANK to deposit money. / I sat by the river BANK.",
    "Syntactic (structural) ambiguity": "I saw the man with the telescope.",
    "Semantic ambiguity": "Visiting relatives can be boring. (who is visiting whom?)",
    "Pragmatic ambiguity": "Can you pass the salt? (literal question vs. request)",
}
for k, v in examples.items():
    print(f"{k}:\n  {v}\n")


In [ ]:
# ============================================================
# PART D: Word Sense Disambiguation -- hand-built Lesk, NLTK Lesk, Naive Bayes
# ============================================================
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk
import nltk

def hand_built_lesk(word, sentence):
    context = set(nltk.word_tokenize(sentence.lower()))
    best_sense, best_overlap = None, -1
    for sense in wn.synsets(word):
        signature = set(nltk.word_tokenize(sense.definition().lower()))
        for ex in sense.examples():
            signature |= set(nltk.word_tokenize(ex.lower()))
        overlap = len(signature & context)
        if overlap > best_overlap:
            best_overlap, best_sense = overlap, sense
    return best_sense

sentence = "I went to the bank to deposit money"
print("Hand-built Lesk:", hand_built_lesk("bank", sentence))
print("NLTK Lesk:", lesk(nltk.word_tokenize(sentence), "bank"))

# --- Naive Bayes sense classifier (toy dataset) ---
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

train_sents = [
    ("I deposited cash at the bank", "financial"),
    ("The loan was approved by the bank", "financial"),
    ("We sat on the river bank and fished", "geographical"),
    ("The bank of the river was muddy", "geographical"),
]
X_train = [s for s, _ in train_sents]
y_train = [l for _, l in train_sents]

vec = CountVectorizer()
X_vec = vec.fit_transform(X_train)
clf = MultinomialNB().fit(X_vec, y_train)

test = ["He walked along the bank near the water"]
pred = clf.predict(vec.transform(test))
print("\nNaive Bayes WSD prediction for:", test[0], "->", pred[0])
